## Load & print sample

In [ ]:
import pickle
import os
import pandas as pd
from IPython.display import display

sample_label_path = "/data/blanka/DATASETS/SPN/YOLOv8x/validation/label"
sample_data_path = "/data/blanka/DATASETS/SPN/YOLOv8x/validation/data"

sample = "122.pkl"

label_df = pd.read_pickle(os.path.join(sample_label_path, sample))
print(label_df.to_string())

state_df = pd.read_pickle(os.path.join(sample_data_path, sample))
print(state_df.to_string())

## Encode & decode state/label

In [22]:
from state_predictor.coder import Coder

coder = Coder(state_df, label_df, alpha_range=(0, 2.2))
encoded_state = coder.encode_state(state_df)
encoded_label = coder.encode_label(label_df)

## Normalize & denormalize label

In [23]:
from state_predictor.utils import normalize, denormalize

dmap = -0.2
range = (0, 1)

norm_dmap = normalize(dmap, range)
print(norm_dmap)
denorm_dmap = denormalize(norm_dmap, range)
print(denorm_dmap)

-1.4
-0.19999999999999996


## Test SPN model (predict)

In [24]:
from utils.config_parser import ConfigParser
from src.model.spn_handler import SPNHandler

# Read and save config file
conf = ConfigParser.read("config/spn.ini")

# load or define SPN model
spn_handler = SPNHandler(conf, run_name="20250328_235625_398629_fifty50")
spn_handler.create(is_pretrained=True)
pred_spars, pred_dmap = spn_handler.predict(encoded_state.unsqueeze(dim=0))

# Denormalize the encoded label
decoded_spars, decoded_dmap = denormalize(encoded_label, value_range=(0, 1))

# Calculate spacing based on the longest label
spacing = max(len(f"{decoded_dmap:.4f}"), len(f"{pred_dmap:.4f}"), len(f"{decoded_spars:.4f}"), len(f"{pred_spars:.4f}"))

# Print the values in the specified format with equal spacing
print(f"dmap:\n  gt:   {decoded_dmap:>{spacing}.4f}\n  pred: {pred_dmap:>{spacing}.4f}")
print(f"\nspars:\n  gt:   {decoded_spars:>{spacing}.4f}\n  pred: {pred_spars:>{spacing}.4f}")



dmap:
  gt:   0.6056
  pred: 0.1390

spars:
  gt:   0.0487
  pred: 0.0954


/home/blanka/Multi-Domain-Pruning/src/model/spn_handler.py:206: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location='cpu')
